# Validated batch inference

This notebook demonstrates the reusable implementation from `python/batch_inference.py`. The notebook does not duplicate production logic: feature engineering, input validation, scaling and prediction all have one shared source of truth.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

python_dir = str(PROJECT_ROOT / 'python')
if python_dir not in sys.path:
    sys.path.insert(0, python_dir)

from batch_inference import InputValidationError, engineer_features, run_batch

## Stable machine-type encoding

`Type_L` and `Type_M` are created explicitly. Their presence no longer depends on which machine types happen to occur in one batch.

In [2]:
single_type_batch = pd.read_csv(PROJECT_ROOT / 'data/test/valid_input.csv').iloc[[0]]
engineer_features(single_type_batch)[['Type', 'Type_L', 'Type_M']]

,Type,Type_L,Type_M
0,L,1,0


## Valid batch

The report contains the source record identifier, failure probability, binary decision, the explicit threshold and a content-based model version.

In [3]:
predictions = run_batch(
    PROJECT_ROOT / 'data/test/valid_input.csv',
    threshold=0.30,
)
predictions

,record_index,probability,prediction,threshold,model_version
0,0,0.000680,0,0.3,sha256:1af454a5d425bbe3e85e9f5a031c81ace392a48...
1,1,0.002840,0,0.3,sha256:1af454a5d425bbe3e85e9f5a031c81ace392a48...
2,2,0.084365,0,0.3,sha256:1af454a5d425bbe3e85e9f5a031c81ace392a48...


## Invalid batch

Invalid input is rejected before scaling or prediction. This example intentionally omits `Torque [Nm]`.

In [4]:
try:
    run_batch(PROJECT_ROOT / 'data/test/missing_column.csv', threshold=0.30)
except InputValidationError as error:
    print(error)

Input is missing required columns: ['Torque [Nm]'].
